In [3]:
import sys
from torch.profiler import profile, ProfilerActivity, record_function

TDECOMP_PATH = '..'
if not TDECOMP_PATH in sys.path:
    sys.path.append(TDECOMP_PATH)

Берём SmallLM 

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "arnir0/Tiny-LLM"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,  use_fast=False)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)


Loading weights: 100%|██████████| 12/12 [00:00<00:00, 510.00it/s, Materializing param=model.norm.weight]                            


In [5]:
# train_imdb.py
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    pipeline
)
from datasets import load_dataset
import torch
# from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np

# model_name = "Qwen/Qwen2-0.5B-Instruct"
model_name = 'arnir0/Tiny-LLM'
dataset_name = "imdb"
# output_dir = "./qwen2-0.5b-imdb-finetuned"
output_dir = './tiny-llm'
max_length = 512  # Maximum context length for each sample

# Use 4-bit quantization to drastically reduce memory usage
use_4bit = False
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

# LoRA configuration for Parameter-Efficient Fine-Tuning
lora_r = 64
lora_alpha = 16
lora_dropout = 0.1

# Training arguments
num_train_epochs = 3
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 4
learning_rate = 2e-4
logging_steps = 10
save_steps = 500

print("Loading model and tokenizer...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)
# Set padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization if enabled
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
if use_4bit:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=use_4bit,
        bnb_4bit_quant_type=bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=use_nested_quant,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",  # Automatically places layers on available GPUs
        trust_remote_code=True
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cpu", #auto
        torch_dtype=torch.float32,
        trust_remote_code=True
    )

# # Option 1: Tiny Shakespeare (literary text)
# dataset = load_dataset("tiny_shakespeare", split="train[:5%]")  # First 5%

# Option 2: CNN Daily Mail (news summaries) - smaller subset
# dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:100]")

# Option 3: Wikitext (Wikipedia articles) - small subset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1000]")

# Option 4: Twitter Complaints (short text)
# dataset = load_dataset("twitter_complaints", split="train[:200]")

# Option 5: AG News (news articles)
# dataset = load_dataset("ag_news", split="train[:100]")

print(f"Dataset size: {len(dataset)}")
print(f"Dataset features: {dataset.features}")

# Preprocess the dataset based on its structure
def preprocess_dataset(examples):
    """Extract text from different dataset formats"""
    if 'text' in examples:
        return {"text": examples["text"]}
    elif 'article' in examples:  # CNN Daily Mail
        return {"text": examples["article"]}
    elif 'content' in examples:  # Some datasets
        return {"text": examples["content"]}
    elif 'sentence' in examples:  # Some sentence datasets
        return {"text": examples["sentence"]}
    else:
        # Try to use the first string column
        for key, value in examples.items():
            if isinstance(value[0], str):
                return {"text": examples[key]}
        return {"text": [str(x) for x in examples[list(examples.keys())[0]]]}

# Apply preprocessing
dataset = dataset.map(preprocess_dataset, batched=True)

# Filter out empty texts
dataset = dataset.filter(lambda example: len(example["text"].strip()) > 0)

# Tokenization function
def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=128,  # Reduced for tiny model
        return_tensors="pt"
    )
    tokenized["labels"] = tokenized["input_ids"].clone()
    return tokenized

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Split dataset
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")


# This will dynamically pad the batches during training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We are doing causal LM, not masked LM
)

# Load accuracy metric
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Shift labels and predictions for causal LM (next token prediction)
    # Predictions are for the next token, so we shift labels accordingly
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    
    # Flatten the tokens and get predictions
    predictions = np.argmax(shift_logits, axis=-1).flatten()
    labels = shift_labels.flatten()
    
    # Calculate accuracy, ignoring padding tokens (where label = -100)
    mask = labels != -100
    predictions = predictions[mask]
    labels = labels[mask]
    return accuracy_metric.compute(predictions=predictions, references=labels)

# ----------------------------
# 7. Training Arguments
# ----------------------------
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    # logging_steps=logging_steps,
    logging_steps=5,
    save_steps=save_steps,
    eval_strategy="steps",
    eval_steps=5,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="mlflow",  # Disable external logging like Weights & Biases for simplicity
    fp16=False,  # Use mixed precision training
)

#TODO Почитать про mlflow
#освежить llm, transformers, Trainer и оптимизатор в этой цепочке
#запустить

Loading model and tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 12/12 [00:00<00:00, 484.41it/s, Materializing param=model.norm.weight]                            


Dataset size: 1000
Dataset features: {'text': Value('string')}
Training samples: 517
Validation samples: 130


In [6]:
print(len(dataset))
dataset["text"][2]

647


" The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Ozawa . A large team of writers handled the script . The game 's opening theme was sung by May 'n . \n"

In [7]:
print(train_test_split["train"]['text'][2])
print(train_test_split["train"]["input_ids"][2])
print(train_test_split["train"]["attention_mask"][2], len(train_test_split["train"]["attention_mask"][2]))
print(train_test_split["train"]["labels"][2])

 Fernandez 's first release of 2013 was Race 2 , an ensemble action thriller ( alongside Saif Ali Khan , John Abraham , Deepika Padukone , Ameesha Patel , and Anil Kapoor ) ) , described as the " cinematic equivalent of a trashy novel " by critic Rajeev Masand . She played Omisha , a femme fatale , a role which required her learn fencing and some acrobatics . The film emerged as a commercial success , with the domestic gross of more than ₹ 1 billion ( US $ 15 million ) . In a particularly scathing review , Saibal Chatterjee of NDTV wrote that both Fernandez and Padukone " strut around like wound @-@ up automatons that are all decked @-@ up but have nowhere to go . " Fernandez also appeared in an item number ( music video ) titled " Jaadu Ki Jhappi " for Prabhu Deva 's Ramaiya Vasta Vaiya . 

[1, 29871, 7139, 4182, 29920, 525, 29879, 937, 6507, 310, 29871, 29906, 29900, 29896, 29941, 471, 23613, 29871, 29906, 1919, 385, 21285, 3158, 1468, 5495, 313, 19963, 5701, 361, 10785, 18915, 1919,

In [8]:
data_collator

DataCollatorForLanguageModeling(tokenizer=TokenizersBackend(name_or_path='arnir0/Tiny-LLM', vocab_size=32000, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
), mlm=False, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_tensors='pt', seed=None)

In [9]:
from tdecomp.grad_proj.tensorgrad.config import TensorGRaDConfig, DataConfig, OptimizerConfig
import tdecomp.matrix.functional as F 

2026-02-17 15:43:50.565123: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-17 15:43:50.673320: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-17 15:43:52.536937: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


ParallelTG, ULTG - это вспомогательные классы-фабрики, убирающие лишние настройки, чтобы было проще. 
Если понадобится более тонкая настройка -- смотрите TensorGRaDConfig и передавайте соответствующие поля (там есть **kwargs) 

In [10]:
from tdecomp.grad_proj.tensorgrad.prepared_tg import ParallelTG, ULTG
a, b = ParallelTG(model, 
                    F.cur, # svd_type можете зарегистрировать свои разложения матричные в модуле F, 
                    # а также просто `truncated_svd` и `randomized_svd`
                    (8, 0.5), # первый и второй ранги. может быть Number, тогда ранги ставятся одинкаовыми.
                    # 0 < float < 1 интерпретируется как доля параметров, int - непосредственно ранг
                                  n_train=len(train_dataset),
                                  batch_size=per_device_train_batch_size,
                                  scheduler='StepLR'
                                  )
    # compute_metrics=compute_metrics, # Uncomment for per-epoch metrics (slower)
a

TensorGRaD (
Parameter Group 0
    betas: (0.9, 0.999)
    correct_bias: True
    eps: 1e-06
    initial_lr: 0.0001
    lr: 0.0001
    weight_decay: 0.0

Parameter Group 1
    batch_size: 4
    betas: (0.9, 0.999)
    correct_bias: True
    dim: 2
    enforce_full_complex_precision: False
    epochs: 100
    eps: 1e-06
    galore_2d_proj_type: left
    initial_lr: 0.0001
    lambda_sparse: 0.05
    lr: 0.0001
    n_iter_max_tucker: 10
    optimizer_type: tensorgrad_sum
    proj_type: low_rank
    rank: 8
    reset_sparse_optimizer_states: False
    scale: 1.0
    scale_by_mask_ratio: True
    scheduler_T_max: 100
    second_proj_type: unstructured_sparse
    second_rank: 0.5
    second_scale: 1.0
    second_scale_by_mask_ratio: False
    second_sparse_ratio: 0.25
    second_sparse_type: topk
    sparse_ratio: 0.1
    sparse_type: topk
    svd_type: <function cur at 0x7683b349b2e0>
    training_samples: 517
    tucker_warm_restart: True
    type: galore
    update_proj_gap: 100
    upda

In [11]:
print(len(a.param_groups))
print(a.param_groups[0])
print(a.param_groups[1])

2
{'params': [Parameter containing:
tensor([0.4922, 0.5806, 0.4700, 0.6748, 0.5957, 0.6226, 0.5928, 0.6514, 0.6934,
        0.4734, 0.4487, 0.5854, 0.6133, 0.6318, 0.6567, 0.5454, 0.6196, 0.5698,
        0.4033, 0.5898, 0.6528, 0.6772, 0.4329, 0.5757, 0.7456, 0.4792, 0.5093,
        0.5332, 0.6230, 0.6372, 0.6768, 0.4626, 0.5957, 0.6001, 0.4817, 0.6338,
        0.5752, 0.7622, 0.5376, 0.6162, 0.7480, 0.5151, 0.4050, 0.5835, 0.5371,
        0.6582, 0.6089, 0.6401, 0.4761, 0.7310, 0.6582, 0.4089, 0.5083, 0.5762,
        0.8237, 0.6953, 0.6162, 0.7754, 0.6162, 0.3540, 0.6113, 0.5474, 0.4255,
        0.6831, 0.5645, 0.6528, 0.6138, 0.6226, 0.4810, 0.5273, 0.3923, 0.6729,
        0.7944, 0.6323, 0.5259, 0.5107, 0.7437, 0.8350, 0.5430, 0.6792, 0.5435,
        0.5249, 0.7271, 0.6602, 0.5342, 0.4148, 0.5161, 0.6011, 0.6313, 0.4138,
        0.6172, 0.8120, 0.7183, 0.6396, 0.6636, 0.5029, 0.5884, 0.4946, 0.5020,
        0.5796, 0.6729, 0.6577, 0.5337, 0.4790, 0.5796, 0.5352, 0.5679, 0.6118,
    

In [12]:
[print(tens.shape) for tens in a.param_groups[0]['params']]
print(len(a.param_groups[0]['params']))

torch.Size([192])
torch.Size([192])
torch.Size([192])
3


In [13]:
[print(tens.shape) for tens in a.param_groups[1]['params']]
print(len(a.param_groups[1]['params']))

torch.Size([32000, 192])
torch.Size([192, 192])
torch.Size([96, 192])
torch.Size([96, 192])
torch.Size([192, 192])
torch.Size([1024, 192])
torch.Size([1024, 192])
torch.Size([192, 1024])
torch.Size([32000, 192])
9


In [14]:
from tdecomp.grad_proj.tensorgrad.prepared_tg import ParallelTG, ULTG

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    optimizers=ParallelTG(model, #TODO check that 0.5 не попадает в sparse_type а попадает в second_rank и затем никуда 
                    F.cur, # svd_type можете зарегистрировать свои разложения матричные в модуле F, 
                    # а также просто `truncated_svd` и `randomized_svd`
                    (8, 0.5), # первый и второй ранги. может быть Number, тогда ранги ставятся одинкаовыми.
                    # 0 < float < 1 интерпретируется как доля параметров, int - непосредственно ранг
                                  n_train=len(train_dataset),
                                  batch_size=per_device_train_batch_size,
                                  scheduler='StepLR'
                                  ),
    # compute_metrics=compute_metrics, # Uncomment for per-epoch metrics (slower)
)

#TODO FIX!


In [15]:
train_loader = trainer.get_train_dataloader()
batch = next(iter(train_loader))
print(tokenizer.decode([2, 1, 29871, 13, 450], skip_special_tokens=False))
print(tokenizer.decode(batch['input_ids'][2]))
print(batch['input_ids'][0].shape)
print(batch["input_ids"].shape)
print(batch["input_ids"])

</s><s> 
 The
<s>  = = In the Union Navy = = 
</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s>
torch.Size([128])
torch.Size([4, 128])
tensor([[    1, 29871,   450, 11143, 28908,   379, 12050,  6726,  2056,  6054,
           347,  1919, 29871, 29896, 29929, 29941, 29941, 29871,    13,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
             2,     2,     2,     2,     2,     2,     

In [21]:
! timeout 2 bash -lc 'echo > /dev/tcp/172.19.0.1/5000' && echo OK || echo FAIL

OK


In [22]:
! timeout 2 bash -lc 'echo > /dev/tcp/localhost/5000' && echo OK || echo FAIL

bash: connect: Connection refused
bash: line 1: /dev/tcp/localhost/5000: Connection refused
FAIL


In [34]:
! timeout 2 bash -lc 'echo > /dev/tcp/mlflow-network/5000' && echo OK || echo FAIL

bash: line 1: mlflow-network: Temporary failure in name resolution
bash: line 1: /dev/tcp/mlflow-network/5000: Invalid argument
FAIL


In [94]:
import mlflow
import mlflow.pytorch
from mlflow.server import get_app_client
import os

mlflow.set_tracking_uri("http://172.19.0.1:5000")
mlflow.set_experiment("my-first-experiment-glazkov")
# client = get_app_client("basic-auth", "http://172.19.0.1:5000")
# client.create_user(username="minio", password="minio123")
# # mlflow.export

os.environ["AWS_ACCESS_KEY_ID"] = "minio"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minio123"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = f"http://172.19.0.1:9000" #https://github.com/mlflow/mlflow/issues/2150


with mlflow.start_run(run_name="particular-run-name"):
    # Log model with signature and input example
    

    mlflow.log_params({'param1': 'lal', 'my_lal_param': 'lol'})

    mlflow.log_metric("accuracy", 150)

    # Optional: Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    mlflow.pytorch.log_model(
        model,
        name="pytorch_model_example2_glazkov",
    )

    model5 = RandomForestRegressor()
    mlflow.sklearn.log_model(
        model5,
        name="rand_forest",
    )

2026/02/17 18:01:39 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/02/17 18:01:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/02/17 18:01:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
/tdecomp/.venv/lib/python3.12/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative

🏃 View run particular-run-name at: http://172.19.0.1:5000/#/experiments/58/runs/871173a5c8db4fee92da4954c3d9588a
🧪 View experiment at: http://172.19.0.1:5000/#/experiments/58


In [50]:
top_models = mlflow.search_logged_models(
    experiment_ids=["58"],
)
top_models

,artifact_location,creation_timestamp,experiment_id,last_updated_timestamp,metrics,model_id,model_type,name,params,source_run_id,status,status_message,tags
0,s3://mlflow/58/models/m-5096569ed5a54301a9523d...,1771348062617,58,1771348071857,"[<Metric: dataset_digest=None, dataset_name=No...",m-5096569ed5a54301a9523ddaffa9286a,,pytorch_model_example2_glazkov,"{'my_lal_param': 'lol', 'param1': 'lal'}",3457d9817ca84e01a09d9f4eee7d660c,READY,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
1,s3://mlflow/58/models/m-4929d3f22ed34e00b18ec4...,1771346306994,58,1771346317619,[],m-4929d3f22ed34e00b18ec4c4bef8d675,,pytorch_model_example_glazkov,{},780eaea33138427f81eea00924473b6f,READY,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
2,s3://mlflow/58/models/m-21415c0700334ed0ba4e90...,1771345730482,58,1771345739271,[],m-21415c0700334ed0ba4e902df23c3eaf,,pytorch_model_example_glazkov,{},15b798a6e8a54405938eda034c82b817,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
3,s3://mlflow/58/models/m-027ad243e0624e25a38850...,1771345629220,58,1771345640634,[],m-027ad243e0624e25a38850dab86ca5ed,,pytorch_model_example_glazkov,{},c7f350d3c35d49a9ac533fa661636043,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
4,s3://mlflow/58/models/m-6615fc94a59444b992ad1c...,1771343278891,58,1771343290470,[],m-6615fc94a59444b992ad1ced7b26b43b,,pytorch_model_example_glazkov,{},c83c79298d4041d083bb5ce7f6d53419,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
5,s3://mlflow/58/models/m-0a564d14be104241acbe9d...,1771343045536,58,1771343057692,[],m-0a564d14be104241acbe9da1c5293795,,pytorch_model_example_glazkov,{},730e7eab71d14154bd1aaffceeb03002,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
6,s3://mlflow/58/models/m-97311726b55c40fda38ef8...,1771329658209,58,1771329669932,[],m-97311726b55c40fda38ef8571c7c1828,,pytorch_model_example_glazkov,{},8d8b455e0192432696083a458712a3e6,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
7,s3://mlflow/58/models/m-e6baf8d96b454cd59004c9...,1771329172960,58,1771329184742,[],m-e6baf8d96b454cd59004c9f3be1e3b92,,pytorch_model_example_glazkov,{},d5b451204f9b41a49f47fa71d6f54c29,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
8,s3://mlflow/58/models/m-1b04ece8ec1548699bf694...,1771328707331,58,1771328716240,[],m-1b04ece8ec1548699bf6942d3d9b1c27,,pytorch_model_example_glazkov,{},976ca2c5d3ea49c4b11b4b1f8ead3ec8,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."
9,s3://mlflow/58/models/m-9cd3945c34b5492189ebe5...,1771328470400,58,1771328480405,[],m-9cd3945c34b5492189ebe5a8b6aae860,,pytorch_model_example_glazkov,{},242ec1cdc6b94094ade96559cd0f0e92,FAILED,,"{'mlflow.source.name': 'tensorgrad_run.ipynb',..."


In [98]:
# model5 = mlflow.sklearn.load_model("runs:/871173a5c8db4fee92da4954c3d9588a/rand_forest")

In [35]:
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

In [100]:
print("Starting training...")
with mlflow.start_run(run_name="another-run-train"):
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], 
                #  with_stack=True, 
                record_shapes=True,
                #experimental_config=torch._C._profiler._ExperimentalConfig(verbose=True),
                ) as profiler:
        with record_function("Record Train!"):
            trainer.train()

2026/02/17 18:05:36 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Starting training...
### Using Composite Projector Configuration ###
    => Swapping projectors to ensure smaller one is first
    => Sizes after swap: first=0.25, second=8.0
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x7685ab3857f0>
UnstructuredSparseProjector initialized with sparse_ratio=0.25, sparse_type=topk, scale_by_mask_ratio=False
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configuration ###
    => Swapping projectors to ensure smaller one is first
    => Sizes after swap: first=0.25, second=8.0
Update gap scheduler: <tdecomp.grad_proj.tensorgrad.projectors.update_gap_scheduler.UpdateGapScheduler object at 0x7685aade6360>
UnstructuredSparseProjector initialized with sparse_ratio=0.25, sparse_type=topk, scale_by_mask_ratio=False
    => First projector: unstructured_sparse
    => Second projector: low_rank
### Using Composite Projector Configurati

/tdecomp/.venv/lib/python3.12/site-packages/tensorly/backend/__init__.py:202: UserWarning: An output with one or more elements was resized since it had shape [8, 192], which does not match the required output shape [32000, 192]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /pytorch/aten/src/ATen/native/Resize.cpp:31.)
  return getattr(
/tdecomp/.venv/lib/python3.12/site-packages/tensorly/backend/__init__.py:202: UserWarning: An output with one or more elements was resized since it had shape [8, 192], which does not match the required output shape [192, 192]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggere

Step,Training Loss,Validation Loss
5,5.446531,5.394906
10,5.295858,5.261163
15,5.187171,5.155357
20,5.114579,5.068389
25,5.016305,4.997982
30,4.870807,4.946933
35,4.966872,4.942661
40,4.877662,4.938423
45,4.916333,4.934189
50,4.997284,4.930018


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.20it/s]
2026/02/17 18:06:28 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/02/17 18:06:28 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run another-run-train at: http://172.19.0.1:5000/#/experiments/58/runs/585903eda4a24a3d95c8c6e8db5915fc
🧪 View experiment at: http://172.19.0.1:5000/#/experiments/58


In [ ]:
torch.cuda.device_count()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 192)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=192, out_features=192, bias=False)
          (k_proj): Linear(in_features=192, out_features=96, bias=False)
          (v_proj): Linear(in_features=192, out_features=96, bias=False)
          (o_proj): Linear(in_features=192, out_features=192, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=192, out_features=1024, bias=False)
          (up_proj): Linear(in_features=192, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=192, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((192,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((192,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((192,), eps=1e-05)
    (rotary_emb): LlamaRotaryEm

In [ ]:
print(model)
len(list(model.parameters()))    
for p in model.parameters():
    print(p)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 192)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=192, out_features=192, bias=False)
          (k_proj): Linear(in_features=192, out_features=96, bias=False)
          (v_proj): Linear(in_features=192, out_features=96, bias=False)
          (o_proj): Linear(in_features=192, out_features=192, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=192, out_features=1024, bias=False)
          (up_proj): Linear(in_features=192, out_features=1024, bias=False)
          (down_proj): Linear(in_features=1024, out_features=192, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((192,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((192,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((192,), eps=1e-05)
    (rotary_emb): LlamaRotaryEm

In [ ]:
torch.embedding()


In [113]:
for name, module in model.named_modules():
    if (len(list(module.children())) == 0):
        print(name)
        if (hasattr(module, 'weight')):
            print("weight - true")
        else: print('weight - false')

model.embed_tokens
weight - true
model.layers.0.self_attn.q_proj
weight - true
model.layers.0.self_attn.k_proj
weight - true
model.layers.0.self_attn.v_proj
weight - true
model.layers.0.self_attn.o_proj
weight - true
model.layers.0.mlp.gate_proj
weight - true
model.layers.0.mlp.up_proj
weight - true
model.layers.0.mlp.down_proj
weight - true
model.layers.0.mlp.act_fn
weight - false
model.layers.0.input_layernorm
weight - true
model.layers.0.post_attention_layernorm
weight - true
model.norm
weight - true
model.rotary_emb
weight - false
lm_head
weight - true


In [166]:
model.model.layers[0].mlp

LlamaMLP(
  (gate_proj): Linear(in_features=192, out_features=1024, bias=False)
  (up_proj): Linear(in_features=192, out_features=1024, bias=False)
  (down_proj): Linear(in_features=1024, out_features=192, bias=False)
  (act_fn): SiLUActivation()
)

In [151]:
# model.named_children()
# model.children()

In [122]:
print(len(list(model.parameters(recurse=False))))
print(len(list(model.parameters(recurse=True))))

0
12


In [137]:
type(list(model.named_modules())[2][1].parameters(recurse=False).__iter__().__next__())

torch.nn.parameter.Parameter

In [142]:
print(len(model.state_dict()))
model.state_dict()

12


OrderedDict([('model.embed_tokens.weight',
              tensor([[ 5.1498e-04,  8.9216e-04, -9.9945e-04,  ..., -4.0936e-04,
                        7.7057e-04, -1.8244e-03],
                      [-2.7274e-03, -3.7239e-04,  5.9496e-04,  ...,  1.9564e-03,
                       -2.4081e-04,  1.3295e-03],
                      [-1.2164e-01, -2.3468e-02, -2.1957e-02,  ...,  5.3902e-03,
                        6.3896e-03, -3.7174e-03],
                      ...,
                      [-9.5367e-03, -1.9321e-03, -1.4229e-02,  ..., -1.4503e-02,
                       -1.3895e-03,  1.0963e-02],
                      [ 1.4544e-03, -5.3291e-03,  5.6076e-03,  ..., -2.0676e-03,
                        2.1148e-04,  9.4299e-03],
                      [-5.4283e-03,  6.3539e-05,  1.1017e-02,  ..., -2.2106e-03,
                       -3.1769e-02,  3.9093e-02]], device='cuda:0')),
             ('model.layers.0.self_attn.q_proj.weight',
              tensor([[ 0.0065, -0.0371, -0.0055,  ..., -0.0308, -0.

In [1]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
comment="free gpu"
profiler.export_chrome_trace(f"profile_{timestamp}_{comment}.json")

NameError: name 'profiler' is not defined

/tdecomp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/02/17 11:26:47 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (loggin

🏃 View run painted-sponge-753 at: http://172.19.0.1:5000/#/experiments/0/runs/12f93f07c5e444c3a225a565445f9257
🧪 View experiment at: http://172.19.0.1:5000/#/experiments/0


NameError: name 'model' is not defined